In [13]:
import pandas as pd
import numpy as np

from catboost import CatBoostRegressor

In [7]:
path = 'processed/v1/train_wide_cleaned.csv'

df = pd.read_csv(path)

df.head()

,user_id,sex,age,is_new_customer,seniority_months,region_code,region_name,income,segment,dep-7,...,dep-2,loan-2,card-4,loan-1,srv-3,loan-5,biz-4,dep-6,loan-4,card-3
0,657788,F,42.0,0.0,114.0,28.0,MADRID,132559.35,INDIVIDUALS,0,...,0,0,0,0,0,0,0,0,0,0
1,657795,M,44.0,0.0,114.0,26.0,"RIOJA, LA",81399.57,INDIVIDUALS,0,...,0,0,0,0,0,0,0,0,0,0
2,657790,M,42.0,0.0,114.0,48.0,BIZKAIA,NaN,INDIVIDUALS,0,...,0,0,0,0,0,1,0,0,1,1
3,657794,F,49.0,0.0,114.0,8.0,BARCELONA,102189.00,VIP,0,...,0,0,0,0,0,0,0,0,0,0
4,657789,M,36.0,0.0,91.0,28.0,MADRID,153725.49,VIP,0,...,0,0,0,0,0,1,0,0,0,0


In [8]:
RANDOM_SEED = 52
rng = np.random.default_rng(RANDOM_SEED)

In [9]:
def get_result_cols(path):
    df = pd.read_csv(path)

    cols = df.columns.tolist()

    cols.remove("income")

    cols.append("income_generated")
    cols.append("income_filled")


    return cols

In [10]:
def make_age_group(age):
    if age < 18:
        return 'UNDER_18'
    if age <= 24:
        return '18_24'
    if age <= 34:
        return '25_34'
    if age <= 44:
        return '35_44'
    if age <= 54:
        return '45_54'
    if age <= 64:
        return '55_64'
    return '65_PLUS'

In [11]:
def prepare_income_features(df):
    df = df.copy()

    PREFIXES = ["dep-", "card-", "rko-", "loan-", "srv-", "biz-"]

    product_cols = [
        col for col in df.columns
        if col.startswith(tuple(PREFIXES))
    ]

    # create age groups
    df["age_group"] = df["age"].apply(make_age_group)

    # product flags
    for col in product_cols:
        df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0).astype("float64")

    dep_cols = [col for col in product_cols if col.startswith("dep-")]
    card_cols = [col for col in product_cols if col.startswith("card-")]
    rko_cols = [col for col in product_cols if col.startswith("rko-")]
    loan_cols = [col for col in product_cols if col.startswith("loan-")]
    srv_cols = [col for col in product_cols if col.startswith("srv-")]
    biz_cols = [col for col in product_cols if col.startswith("biz-")]

    # aggregate product features
    df["products_count"] = df[product_cols].sum(axis=1) if product_cols else 0.0

    df["dep_count"] = df[dep_cols].sum(axis=1) if dep_cols else 0.0
    df["card_count"] = df[card_cols].sum(axis=1) if card_cols else 0.0
    df["rko_count"] = df[rko_cols].sum(axis=1) if rko_cols else 0.0
    df["loan_count"] = df[loan_cols].sum(axis=1) if loan_cols else 0.0
    df["srv_count"] = df[srv_cols].sum(axis=1) if srv_cols else 0.0
    df["biz_count"] = df[biz_cols].sum(axis=1) if biz_cols else 0.0

    df["has_dep"] = (df["dep_count"] > 0).astype(int)
    df["has_card"] = (df["card_count"] > 0).astype(int)
    df["has_rko"] = (df["rko_count"] > 0).astype(int)
    df["has_loan"] = (df["loan_count"] > 0).astype(int)
    df["has_srv"] = (df["srv_count"] > 0).astype(int)
    df["has_biz"] = (df["biz_count"] > 0).astype(int)

    # create product profile
    def product_profile(row):
        if row["products_count"] == 0:
            return "NO_PRODUCTS"

        profile = []

        if row["has_dep"]:
            profile.append("DEP")
        if row["has_card"]:
            profile.append("CARD")
        if row["has_rko"]:
            profile.append("RKO")
        if row["has_loan"]:
            profile.append("LOAN")
        if row["has_srv"]:
            profile.append("SRV")
        if row["has_biz"]:
            profile.append("BIZ")

        return "_".join(profile)

    df["product_profile"] = df.apply(product_profile, axis=1)

    return df, product_cols

In [12]:
def generate_income_from_existing_data(path):
    df = pd.read_csv(path, low_memory=False)

    # prepare features
    df, product_cols = prepare_income_features(df)

    train_mask = df["income"].notna() & (df["income"] > 0)
    train_df = df.loc[train_mask].copy()

    # emissions limit
    lower = train_df["income"].quantile(0.005)
    upper = train_df["income"].quantile(0.900)

    train_df = train_df[
        (train_df["income"] >= lower) &
        (train_df["income"] <= upper)
    ].copy()

    y_train = train_df["income"]

    # categorical features
    cat_features = [
        # base features
        "sex",
        "segment",
        "age_group",
        "product_profile",

        # region features
        "region_code",
        "region_name"
    ]

    # numerical features
    num_features = [
        "age",
        "is_new_customer",
        "seniority_months",
        "products_count",
        "dep_count",
        "card_count",
        "rko_count",
        "loan_count",
        "srv_count",
        "biz_count",
        "has_dep",
        "has_card",
        "has_rko",
        "has_loan",
        "has_srv",
        "has_biz"
    ]

    # product flags
    num_features = num_features + product_cols

    features = cat_features + num_features

    for col in cat_features:
        df[col] = df[col].fillna("UNKNOWN").astype(str)
        train_df[col] = train_df[col].fillna("UNKNOWN").astype(str)

    for col in num_features:
        df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0).astype("float64")
        train_df[col] = pd.to_numeric(train_df[col], errors="coerce").fillna(0).astype("float64")

    X_train = train_df[features]
    X_all = df[features]

    model = CatBoostRegressor(
        loss_function="MAE",
        eval_metric="MAE",
        iterations=1500,
        learning_rate=0.04,
        depth=7,
        l2_leaf_reg=7,
        random_seed=RANDOM_SEED,
        verbose=100,
        allow_writing_files=False
    )

    model.fit(
        X_train,
        y_train,
        cat_features=cat_features
    )

    # deterministic prediction
    pred_all = model.predict(X_all)
    pred_all = np.clip(pred_all, lower, upper)

    df["income_model_prediction"] = np.round(pred_all, 2)

    missing_mask = df["income"].isna() | (df["income"] <= 0)

    df["income_generated"] = np.nan
    df.loc[missing_mask, "income_generated"] = df.loc[
        missing_mask,
        "income_model_prediction"
    ]

    df["income_filled"] = df["income"]
    df.loc[missing_mask, "income_filled"] = df.loc[
        missing_mask,
        "income_generated"
    ]

    df["income_was_generated"] = missing_mask.astype(int)

    # keep result columns
    cols = get_result_cols(path)

    cols = [
        col for col in cols
        if col in df.columns
    ]

    df = df[cols]

    return df, model

In [14]:
output_path = 'processed/v2/train_wide_cleaned_with_income.csv'

df.to_csv(output_path, index=False)